# LeetCode 3615 — Longest Palindromic Path in Graph bằng Python

- Link: https://leetcode.com/problems/longest-palindromic-path-in-graph/
- Độ khó: **Hard**
- Chủ đề: Bitmask DP / tìm kiếm trên trạng thái `(tập đỉnh, đầu trái, đầu phải)`, **mở rộng palindrome từ giữa ra hai phía**, đường đi đơn

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxLen(self, n: int, edges: List[List[int]], label: str) -> int:
```

## 1. Mục tiêu học

1. Palindrome có thể **xây từ tâm**: tâm là một đỉnh (độ dài lẻ) hoặc một cạnh nối hai đỉnh cùng nhãn (độ dài chẵn).
2. Mỗi bước mở rộng thêm **cùng lúc** một đỉnh ở đầu trái và một đỉnh ở đầu phải, hai đỉnh mới **cùng nhãn**, chưa dùng, kề với hai đầu tương ứng.
3. Trạng thái `(mask, i, j)` (tập đỉnh đã dùng, hai đầu); với `n ≤ 14` có tối đa `2^14 · 14 · 14 / 2 ≈ 1,6·10^6` trạng thái.
4. Tăng tốc bằng bitmask: láng giềng chưa dùng có nhãn `c` = `adj[i] & ~mask & lab[c]`.
5. Kiểm chứng bằng liệt kê mọi đường đơn.

## 2. Đề bài & ví dụ

Đồ thị vô hướng `n` đỉnh, đỉnh `i` có nhãn `label[i]`. Đi một **đường đơn** (mỗi đỉnh tối đa một lần), bắt đầu ở đâu cũng được. Tìm độ dài lớn nhất của chuỗi nhãn là palindrome.

**Ví dụ 1:** `n = 3, edges = [[0,1],[1,2]], label = "aba"` → `3`.

**Ví dụ 2:** `n = 3, edges = [[0,1],[0,2]], label = "abc"` → `1`.

**Ví dụ 3:** `n = 4, edges = [[0,2],[0,3],[3,1]], label = "bbac"` → `3` (`0 → 3 → 1` = `"bcb"`).

**Ràng buộc:** `n ≤ 14`, không có cạnh trùng.

In [1]:
from typing import List
from collections import deque
from functools import lru_cache
import random, time

examples = [(3, [[0, 1], [1, 2]], "aba", 3), (3, [[0, 1], [0, 2]], "abc", 1), (4, [[0, 2], [0, 3], [3, 1]], "bbac", 3)]
for n, e, lab, exp in examples:
    print(n, e, lab, "-> kỳ vọng", exp)

3 [[0, 1], [1, 2]] aba -> kỳ vọng 3
3 [[0, 1], [0, 2]] abc -> kỳ vọng 1
4 [[0, 2], [0, 3], [3, 1]] bbac -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Palindrome xây từ tâm

`s` là palindrome ⇔ bỏ ký tự đầu và cuối (nếu bằng nhau) vẫn là palindrome. Đảo lại: từ một palindrome, thêm cùng một ký tự vào **hai đầu** được palindrome dài hơn 2. Tâm: một ký tự (độ dài lẻ) hoặc hai ký tự bằng nhau (độ dài chẵn).

### 3.2. Đường đơn và tập đỉnh

Với đường đơn, cần biết **tập đỉnh đã dùng** để không lặp lại ⇒ bitmask `mask`. Hai đầu hiện tại `i`, `j` (thứ tự không quan trọng ⇒ chuẩn hoá `i ≤ j`).

In [2]:
def bits(m): return [i for i in range(m.bit_length()) if m >> i & 1]
print("mask 0b10110 ->", bits(0b10110), "| popcount =", bin(0b10110).count("1"))

mask 0b10110 -> [1, 2, 4] | popcount = 3


### 3.3. Láng giềng theo nhãn bằng bitmask

`adj[i]` là bitmask láng giềng; `lab[c]` là bitmask các đỉnh nhãn `c`. Ứng viên mở rộng đầu `i` với nhãn `c`: `adj[i] & ~mask & lab[c]` — một phép AND thay cho vòng lặp.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Mọi đường palindrome `v_1 … v_L` có "lõi" ở giữa; đọc từ lõi ra, mỗi bước thêm `v_{t}` ở trái và `v_{L+1−t}` ở phải với cùng nhãn. Vậy tập mọi palindrome = tập mọi trạng thái đạt được từ các tâm qua phép mở rộng. Độ dài palindrome = `popcount(mask)`.

Ví dụ 3: tâm `{3}` (nhãn `c`); mở rộng: láng giềng của 3 là `0` (`b`) và `1` (`b`) — cùng nhãn ⇒ trạng thái `({0,1,3}, 0, 1)`, độ dài 3. ✔

## 5. Cách 1 — Brute Force: liệt kê mọi đường đơn

In [3]:
def brute(n, edges, label):
    adj = [[] for _ in range(n)]
    for u, v in edges: adj[u].append(v); adj[v].append(u)
    best = 1
    def dfs(path, used):
        nonlocal best
        s = "".join(label[x] for x in path)
        if s == s[::-1]: best = max(best, len(path))
        for v in adj[path[-1]]:
            if not used >> v & 1:
                path.append(v); dfs(path, used | 1 << v); path.pop()
    for s in range(n): dfs([s], 1 << s)
    return best

for n, e, lab, exp in examples:
    print(brute(n, e, lab), "kỳ vọng", exp)

3 kỳ vọng 3
1 kỳ vọng 1
3 kỳ vọng 3


**Nút thắt:** số đường đơn có thể tới `n!` (đồ thị đầy đủ `14! ≈ 8,7·10^{10}`).

## 6. Ý tưởng thuật toán / bước đột phá

Tìm kiếm trên đồ thị trạng thái `(mask, i, j)`:

- **Khởi tạo:** `({v}, v, v)` mọi `v`; `({u, v}, u, v)` mọi cạnh có `label[u] = label[v]`.
- **Chuyển:** với mỗi nhãn `c`, `x ∈ adj[i] & ~mask & lab[c]`, `y ∈ adj[j] & ~mask & lab[c]`, `x ≠ y` ⇒ `(mask | x | y, x, y)`.
- Đáp án = `max popcount(mask)`; dừng sớm khi đạt `n`.

Mỗi trạng thái thăm một lần (tập `seen`).

## 7. Các cách tiếp theo

- **Cách 2:** BFS theo **độ dài** (lớp): mọi trạng thái độ dài `L` sinh độ dài `L + 2` — dễ dừng khi không còn lớp mới.
- **Cách 3:** DFS + `seen` + dừng sớm khi đạt `n` (lời giải nộp).
- **Cách 4:** đệ quy có nhớ `f(mask, i, j)` = độ dài lớn nhất đạt được từ trạng thái — kiểu "quy hoạch động trên trạng thái".

### 7.1. Cách 2 — BFS theo lớp độ dài

In [4]:
def prepare(n, edges, label):
    adj = [0] * n
    for u, v in edges: adj[u] |= 1 << v; adj[v] |= 1 << u
    lab = {}
    for i, c in enumerate(label): lab[c] = lab.get(c, 0) | 1 << i
    return adj, lab

def expand(state, adj, lab):
    m, i, j = state; free = ~m; Ai = adj[i] & free; Aj = adj[j] & free
    for lm in lab.values():
        A = Ai & lm; B = Aj & lm
        if not A or not B: continue
        xa = A
        while xa:
            xb = xa & -xa; xa ^= xb; x = xb.bit_length() - 1
            ya = B & ~xb
            while ya:
                yb = ya & -ya; ya ^= yb; y = yb.bit_length() - 1
                yield (m | xb | yb, min(x, y), max(x, y))

def bfs_layers(n, edges, label):
    adj, lab = prepare(n, edges, label)
    layer = {(1 << v, v, v) for v in range(n)}
    even = {((1 << u) | (1 << v), min(u, v), max(u, v)) for u, v in edges if label[u] == label[v]}
    best = 2 if even else 1
    for start in (layer, even):
        cur = set(start)
        while cur:
            best = max(best, bin(next(iter(cur))[0]).count("1"))
            nxt = set()
            for st in cur: nxt.update(expand(st, adj, lab))
            cur = nxt
    return best

for n, e, lab, exp in examples:
    print(bfs_layers(n, e, lab), "kỳ vọng", exp)

3 kỳ vọng 3
1 kỳ vọng 1
3 kỳ vọng 3


**Nhận xét:** các trạng thái trong một lớp có cùng `popcount`, nên lớp cuối khác rỗng cho độ dài lớn nhất (theo tâm lẻ/chẵn).

### 7.2. Cách 3 — DFS + `seen` + dừng sớm (lời giải chính)

In [5]:
def dfs_seen(n, edges, label):
    adj, lab = prepare(n, edges, label)
    seen = set(); stack = []; best = 1
    for v in range(n):
        st = (1 << v, v, v); seen.add(st); stack.append(st)
    for u, v in edges:
        if label[u] == label[v]:
            st = ((1 << u) | (1 << v), min(u, v), max(u, v))
            if st not in seen: seen.add(st); stack.append(st); best = 2
    while stack:
        st = stack.pop(); c = bin(st[0]).count("1")
        if c > best:
            best = c
            if best == n: return n
        for nx in expand(st, adj, lab):
            if nx not in seen: seen.add(nx); stack.append(nx)
    return best

for n, e, lab, exp in examples:
    print(dfs_seen(n, e, lab), "kỳ vọng", exp)

3 kỳ vọng 3
1 kỳ vọng 1
3 kỳ vọng 3


### 7.3. Cách 4 — Đệ quy có nhớ trên trạng thái

In [6]:
def memo_states(n, edges, label):
    adj, lab = prepare(n, edges, label)
    @lru_cache(None)
    def f(m, i, j):
        best = bin(m).count("1")
        for nx in expand((m, i, j), adj, lab): best = max(best, f(*nx))
        return best
    best = max(f(1 << v, v, v) for v in range(n))
    for u, v in edges:
        if label[u] == label[v]: best = max(best, f((1 << u) | (1 << v), min(u, v), max(u, v)))
    return best

for n, e, lab, exp in examples:
    print(memo_states(n, e, lab), "kỳ vọng", exp)

3 kỳ vọng 3
1 kỳ vọng 1
3 kỳ vọng 3


**Lưu ý:** độ sâu đệ quy chỉ `≤ n/2` (mỗi tầng thêm 2 đỉnh) nên an toàn.

## 8. TRACE từng bước

In [7]:
def trace(n, edges, label):
    adj, lab = prepare(n, edges, label)
    names = lambda m: "".join(str(i) for i in bits(m))
    starts = [(1 << v, v, v) for v in range(n)] + [((1 << u) | (1 << v), min(u, v), max(u, v)) for u, v in edges if label[u] == label[v]]
    for st in starts:
        frontier = [st]; depth = 0
        while frontier:
            desc = ", ".join(f"{{{names(m)}}} đầu ({i},{j})" for m, i, j in frontier)
            print(f"tâm {names(st[0])} — bước {depth}: {desc}")
            frontier = list(set(nx for s in frontier for nx in expand(s, adj, lab))); depth += 1

trace(4, [[0, 2], [0, 3], [3, 1]], "bbac")

tâm 0 — bước 0: {0} đầu (0,0)
tâm 1 — bước 0: {1} đầu (1,1)
tâm 2 — bước 0: {2} đầu (2,2)
tâm 3 — bước 0: {3} đầu (3,3)
tâm 3 — bước 1: {013} đầu (0,1)


In [8]:
g = [[0, 1], [1, 2], [2, 3], [3, 4], [1, 5], [5, 3]]
print("đáp án:", dfs_seen(6, g, "abcbax"), "| brute:", brute(6, g, "abcbax"))

đáp án: 5 | brute: 5


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def maxLen(self, n: int, edges: List[List[int]], label: str) -> int:
        adj = [0] * n
        for u, v in edges:
            adj[u] |= 1 << v
            adj[v] |= 1 << u
        lab = {}
        for i, c in enumerate(label):
            lab[c] = lab.get(c, 0) | (1 << i)
        masks = list(lab.values())
        seen = set()
        stack = []
        best = 1
        for v in range(n):
            key = ((1 << v) << 8) | (v << 4) | v
            seen.add(key)
            stack.append(key)
        for u, v in edges:
            if label[u] == label[v]:
                a, b = (u, v) if u < v else (v, u)
                key = (((1 << u) | (1 << v)) << 8) | (a << 4) | b
                if key not in seen:
                    seen.add(key)
                    stack.append(key)
                    best = 2
        while stack:
            key = stack.pop()
            m = key >> 8
            i = (key >> 4) & 15
            j = key & 15
            c = bin(m).count('1')
            if c > best:
                best = c
                if best == n:
                    return n
            free = ~m
            Ai = adj[i] & free
            Aj = adj[j] & free
            if not Ai or not Aj:
                continue
            for lm in masks:
                A = Ai & lm
                B = Aj & lm
                if not A or not B:
                    continue
                xa = A
                while xa:
                    xb = xa & -xa
                    xa ^= xb
                    x = xb.bit_length() - 1
                    ya = B & ~xb
                    while ya:
                        yb = ya & -ya
                        ya ^= yb
                        y = yb.bit_length() - 1
                        a, b = (x, y) if x < y else (y, x)
                        nk = ((m | xb | yb) << 8) | (a << 4) | b
                        if nk not in seen:
                            seen.add(nk)
                            stack.append(nk)
        return best

sol = Solution()
for n, e, lab, exp in examples:
    got = sol.maxLen(n, e, lab)
    print(got, "✔" if got == exp else "✘")

3 ✔
1 ✔
3 ✔


## 10. Vì sao thuật toán đúng

1. **Mọi trạng thái đạt được là một đường palindrome:** bất biến — trạng thái `(m, i, j)` ứng với một đường đơn có tập đỉnh `m`, hai đầu `i`, `j`, chuỗi nhãn palindrome. Khởi tạo đúng; bước mở rộng thêm `x` kề `i`, `y` kề `j`, chưa dùng, khác nhau, cùng nhãn ⇒ đường `x – (đường cũ) – y` vẫn đơn và palindrome.
2. **Mọi đường palindrome đều đạt được:** với đường `v_1 … v_L` palindrome, lõi là `v_{(L+1)/2}` (lẻ) hoặc cặp giữa (chẵn, cùng nhãn và kề nhau ⇒ là một trạng thái khởi tạo). Bóc dần từ ngoài vào cho chuỗi trạng thái hợp lệ ngược lại.
3. Độ dài = `popcount(m)`; lấy max. Dừng sớm khi đạt `n` vì không thể dài hơn.
4. Chuẩn hoá `i ≤ j` không mất thông tin (đường vô hướng, đảo chiều vẫn palindrome).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê đường | `O(n! · n)` xấu nhất | `O(n)` |
| 2. BFS theo lớp | `O(S · n²)` với `S ≤ 2^n n²/2` trạng thái | `O(S)` |
| 3. DFS + `seen` | như trên, thường ít hơn nhờ dừng sớm | `O(S)` |
| 4. Đệ quy có nhớ | như trên | `O(S)` |

Thực nghiệm (đồ thị đầy đủ 14 đỉnh, nhãn `"a"*13 + "b"`): khoảng 2 giây trong Python.

## 12. Bộ test

In [10]:
METHODS = [brute, bfs_layers, dfs_seen, memo_states, lambda n, e, l: Solution().maxLen(n, e, l)]
for n, e, lab, exp in examples:
    assert all(f(n, e, lab) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [11]:
# 12.2 ca biên
assert Solution().maxLen(1, [], "z") == 1
assert Solution().maxLen(2, [[0, 1]], "aa") == 2
assert Solution().maxLen(2, [[0, 1]], "ab") == 1
assert Solution().maxLen(4, [[0, 1], [1, 2], [2, 3], [3, 0]], "abba") == 4         # chu trình
print("Ca biên: OK")

Ca biên: OK


In [12]:
# 12.3 ngẫu nhiên (đồ thị liên thông)
rng = random.Random(3615)
for _ in range(400):
    n = rng.randint(1, 8); edges = [[i, rng.randrange(i)] for i in range(1, n)]
    have = {tuple(sorted(e)) for e in edges}
    for a in range(n):
        for b in range(a + 1, n):
            if (a, b) not in have and rng.random() < 0.3: edges.append([a, b]); have.add((a, b))
    lab = "".join(rng.choice("ab" if rng.random() < 0.5 else "abc") for _ in range(n))
    b = brute(n, edges, lab)
    assert all(f(n, edges, lab) == b for f in METHODS[1:]), (n, edges, lab)
print("400 test ngẫu nhiên: OK")

400 test ngẫu nhiên: OK


In [13]:
# 12.4 thời gian với n = 14
full = [[a, b] for a in range(14) for b in range(a + 1, 14)]
for name, lab in [("đầy đủ, toàn 'a'", "a" * 14), ("đầy đủ, 'ab'*7", "ab" * 7), ("đầy đủ, 'a'*13+'b'", "a" * 13 + "b"), ("đầy đủ, khác nhau", "abcdefghijklmn")]:
    t0 = time.time(); r = Solution().maxLen(14, full, lab); t1 = time.time()
    print(f"{name}: {r} | {t1 - t0:.2f}s")

đầy đủ, toàn 'a': 14 | 0.00s


đầy đủ, 'ab'*7: 13 | 0.86s


đầy đủ, 'a'*13+'b': 13 | 2.17s
đầy đủ, khác nhau: 1 | 0.00s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ xét tâm một đỉnh (quên palindrome độ dài chẵn)

In [14]:
def wrong_odd_only(n, edges, label):
    adj, lab = prepare(n, edges, label); seen = set(); stack = []; best = 1
    for v in range(n): st = (1 << v, v, v); seen.add(st); stack.append(st)              # SAI: thiếu tâm cạnh
    while stack:
        st = stack.pop(); best = max(best, bin(st[0]).count("1"))
        for nx in expand(st, adj, lab):
            if nx not in seen: seen.add(nx); stack.append(nx)
    return best

print("sai:", wrong_odd_only(2, [[0, 1]], "aa"), "| đúng:", brute(2, [[0, 1]], "aa"))

sai: 1 | đúng: 2


### Lỗi 2 — Cho phép `x == y` (cùng một đỉnh được thêm cho cả hai đầu)

In [15]:
def wrong_same_vertex(n, edges, label):
    adj, lab = prepare(n, edges, label); seen = set(); stack = []; best = 1
    for v in range(n): st = (1 << v, v, v); seen.add(st); stack.append(st)
    for u, v in edges:
        if label[u] == label[v]: st = ((1 << u) | (1 << v), min(u, v), max(u, v)); seen.add(st); stack.append(st)
    while stack:
        m, i, j = stack.pop(); best = max(best, len(bits(m)))
        for x in bits(adj[i] & ~m):
            for y in bits(adj[j] & ~m):
                if label[x] == label[y]:                                                  # SAI: thiếu x != y
                    nm = m | 1 << x | 1 << y; st = (nm, min(x, y), max(x, y))
                    if st not in seen: seen.add(st); stack.append(st); best = max(best, len(bits(m)) + 2)
    return best

print("sai:", wrong_same_vertex(3, [[0, 1], [0, 2]], "bab"), "| đúng:", brute(3, [[0, 1], [0, 2]], "bab"))

sai: 4 | đúng: 2


### Lỗi 3 — Mở rộng chỉ một đầu (như tìm đường bình thường) rồi kiểm tra palindrome cuối cùng mà không có `mask`

Không lưu tập đỉnh ⇒ có thể đi lại đỉnh cũ (đường không đơn).

In [16]:
def wrong_no_mask(n, edges, label, max_len=6):
    adj = [[] for _ in range(n)]
    for u, v in edges: adj[u].append(v); adj[v].append(u)
    best = 1
    def go(path):
        nonlocal best
        s = "".join(label[x] for x in path)
        if s == s[::-1]: best = max(best, len(path))
        if len(path) < max_len:
            for v in adj[path[-1]]: go(path + [v])                                       # SAI: cho đi lại đỉnh
    for s in range(n): go([s])
    return best

print("sai:", wrong_no_mask(2, [[0, 1]], "ab"), "| đúng:", brute(2, [[0, 1]], "ab"))

sai: 5 | đúng: 1


### Lỗi 4 — Dùng bộ nhớ `dp[mask][i][j]` dạng list 3 chiều đầy đủ trong Python

`2^14 · 14 · 14 ≈ 3,2·10^6` ô — tốn bộ nhớ và thời gian khởi tạo; tập `seen` chỉ chứa trạng thái **thực sự đạt được** thường nhỏ hơn nhiều.

In [17]:
print("số ô mảng đầy đủ:", (1 << 14) * 14 * 14)

số ô mảng đầy đủ: 3211264


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Palindrome trên đường/đồ thị ⇒ xây **từ giữa ra hai đầu**, trạng thái gồm hai đầu.
- `n ≤ 14–16` + "mỗi đỉnh tối đa một lần" ⇒ bitmask tập đỉnh đã dùng.
- Ghép hai đầu cùng nhãn ⇒ nhóm theo nhãn bằng bitmask để tránh vòng lặp thừa.
- Có cận trên tự nhiên (`n`) ⇒ dừng sớm.

## 15. Biến thể của bài

1. **Cho phép đi lại đỉnh** (walk, không cần đơn): bỏ `mask`, BFS trên cặp `(i, j)` — LC "palindromic walk".
2. **Đồ thị có hướng**: đầu trái đi **ngược** cạnh, đầu phải đi **xuôi** cạnh.
3. **Đếm số đường palindrome**: DP đếm trên cùng đồ thị trạng thái (mỗi đường được sinh từ đúng một tâm).
4. **Cây** thay vì đồ thị tổng quát: đường đơn duy nhất giữa hai đỉnh ⇒ kiểm tra mọi cặp với hashing.

In [18]:
# Biến thể 1: đường đi được lặp đỉnh, giới hạn độ dài tối đa L — có tồn tại palindrome độ dài L không?
def palindromic_walk_lengths(n, edges, label, L):
    adj = [[] for _ in range(n)]
    for u, v in edges: adj[u].append(v); adj[v].append(u)
    odd = {(v, v) for v in range(n)}; even = {(min(u, v), max(u, v)) for u, v in edges if label[u] == label[v]}
    found = {1} if n else set()
    if even: found.add(2)
    for start, base in ((odd, 1), (even, 2)):
        cur = set(start); length = base
        while cur and length + 2 <= L:
            nxt = {(min(x, y), max(x, y)) for i, j in cur for x in adj[i] for y in adj[j] if label[x] == label[y]}
            length += 2
            if nxt: found.add(length)
            cur = nxt
    return sorted(found)

print("độ dài palindrome walk (≤ 7) trên đường 0-1-2 nhãn 'aba':", palindromic_walk_lengths(3, [[0, 1], [1, 2]], "aba", 7))

độ dài palindrome walk (≤ 7) trên đường 0-1-2 nhãn 'aba': [1, 3, 5, 7]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Liệt kê đường | DFS đường đơn | `O(n!)` | Kiểm chứng |
| 2. BFS theo lớp | Mở rộng từ tâm | `O(S n²)` | Dễ hiểu |
| 3. DFS + `seen` | Như 2 + dừng sớm | `O(S n²)` | **Lời giải chuẩn** |
| 4. Đệ quy có nhớ | DP trên trạng thái | `O(S n²)` | Khi cần giá trị theo trạng thái |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** Đường thẳng `0-1-2-3-4`, nhãn `"abcba"`.

**Bài 2 (tính tay).** Hình sao tâm 0 (nhãn `a`) với 4 lá nhãn `b, b, c, c`.

**Bài 3 (điền TODO).** Viết `palindrome_lengths(n, edges, label)` — tập mọi độ dài `L` sao cho tồn tại đường đơn palindrome độ dài đúng `L`.

**Bài 4 (tự viết lại).** Viết lại lời giải trả về luôn **một đường** palindrome dài nhất.

**Bài 5 (mini challenge).** Giải bài trên **đồ thị có hướng** (đường đi theo chiều cạnh).

In [19]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def palindrome_lengths(n, edges, label):
    # TODO: tập popcount của mọi trạng thái đạt được
    pass

In [20]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def max_len_directed(n, edges, label):
    # TODO: đầu trái đi theo cạnh ngược, đầu phải theo cạnh xuôi
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Tâm `2` (`c`). (2) Mở rộng `1, 3` (`b, b`), rồi `0, 4` (`a, a`). (3) `5`.

**Bài 2.** (1) Mọi đường qua tâm có dạng `lá – 0 – lá`. (2) Hai lá cùng nhãn: `b a b` hoặc `c a c`. (3) `3`.

**Bài 3.** (1) Mỗi trạng thái đạt được là một đường palindrome và ngược lại (mục 10). (2) Thu `popcount(mask)` của mọi trạng thái vào một tập. (3) Lưu ý: **đếm số đường** thì không thể chỉ đếm trạng thái — cùng `(mask, i, j)` có thể ứng với nhiều thứ tự đỉnh khác nhau.

**Bài 4.** (1) Lưu cha của mỗi trạng thái. (2) Từ trạng thái tốt nhất, truy vết các cặp `(x, y)` thêm vào. (3) Ghép trái đảo + tâm + phải.

**Bài 5.** (1) Trạng thái `(mask, i, j)` với `i` là đầu (đi vào), `j` là cuối (đi ra). (2) `x` có cạnh `x → i`, `y` có cạnh `j → y`. (3) Tâm chẵn: cạnh `u → v` cùng nhãn.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
assert Solution().maxLen(5, [[0, 1], [1, 2], [2, 3], [3, 4]], "abcba") == 5
assert Solution().maxLen(5, [[0, 1], [0, 2], [0, 3], [0, 4]], "abbcc") == 3 == brute(5, [[0, 1], [0, 2], [0, 3], [0, 4]], "abbcc")
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [22]:
# Bài 3, 4
def palindrome_lengths_sol(n, edges, label):
    adj, lab = prepare(n, edges, label); seen = set(); stack = []
    for v in range(n): st = (1 << v, v, v); seen.add(st); stack.append(st)
    for u, v in edges:
        if label[u] == label[v]: st = ((1 << u) | (1 << v), min(u, v), max(u, v)); seen.add(st); stack.append(st)
    while stack:
        for nx in expand(stack.pop(), adj, lab):
            if nx not in seen: seen.add(nx); stack.append(nx)
    return {bin(m).count("1") for m, _, _ in seen}

def brute_lengths(n, edges, label):
    adj = [[] for _ in range(n)]
    for u, v in edges: adj[u].append(v); adj[v].append(u)
    found = set()
    def dfs(path, used):
        s = "".join(label[x] for x in path)
        if s == s[::-1]: found.add(len(path))
        for v in adj[path[-1]]:
            if not used >> v & 1: dfs(path + [v], used | 1 << v)
    for s0 in range(n): dfs([s0], 1 << s0)
    return found

def longest_path_sol(n, edges, label):
    adj, lab = prepare(n, edges, label); par = {}; stack = []
    for v in range(n): st = (1 << v, v, v); par[st] = None; stack.append(st)
    for u, v in edges:
        if label[u] == label[v]:
            st = ((1 << u) | (1 << v), min(u, v), max(u, v))
            if st not in par: par[st] = None; stack.append(st)
    while stack:
        st = stack.pop()
        for nx in expand(st, adj, lab):
            if nx not in par: par[nx] = st; stack.append(nx)
    cur = max(par, key=lambda s: bin(s[0]).count("1"))
    a, b = cur[1], cur[2]; left, right = [a], [b]            # đầu trái/phải hiện tại (theo hướng đã cố định)
    while par[cur] is not None:
        prev = par[cur]; _, pi, pj = prev
        if adj[a] >> pi & 1 and adj[b] >> pj & 1: na, nb = pi, pj
        else: na, nb = pj, pi
        left.append(na); right.append(nb); a, b = na, nb; cur = prev
    # left: từ ngoài vào trong; right: từ ngoài vào trong. Tâm: phần tử cuối của hai danh sách
    if left[-1] == right[-1]: path = left + right[-2::-1]
    else: path = left + right[::-1]
    return path

rng = random.Random(4)
for _ in range(150):
    n = rng.randint(1, 7); edges = [[i, rng.randrange(i)] for i in range(1, n)]
    have = {tuple(sorted(e)) for e in edges}
    for a in range(n):
        for b in range(a + 1, n):
            if (a, b) not in have and rng.random() < 0.3: edges.append([a, b]); have.add((a, b))
    lab = "".join(rng.choice("ab") for _ in range(n))
    assert palindrome_lengths_sol(n, edges, lab) == brute_lengths(n, edges, lab)
    p = longest_path_sol(n, edges, lab); s = "".join(lab[x] for x in p)
    adjs = {tuple(sorted(e)) for e in edges}
    assert len(p) == brute(n, edges, lab) and s == s[::-1] and len(set(p)) == len(p)
    assert all(tuple(sorted((p[t], p[t + 1]))) in adjs for t in range(len(p) - 1)), (n, edges, lab, p)
print("Bài 3, 4 OK; ví dụ 3:", longest_path_sol(4, [[0, 2], [0, 3], [3, 1]], "bbac"))

Bài 3, 4 OK; ví dụ 3: [0, 3, 1]


In [23]:
# Bài 5: đồ thị có hướng
def max_len_directed_sol(n, edges, label):
    out_ = [0] * n; in_ = [0] * n
    for u, v in edges: out_[u] |= 1 << v; in_[v] |= 1 << u
    seen = set(); stack = []; best = 1
    for v in range(n): st = (1 << v, v, v); seen.add(st); stack.append(st)
    for u, v in edges:
        if label[u] == label[v]:
            st = ((1 << u) | (1 << v), u, v)
            if st not in seen: seen.add(st); stack.append(st); best = 2
    while stack:
        m, i, j = stack.pop(); best = max(best, bin(m).count("1"))
        for x in bits(in_[i] & ~m):
            for y in bits(out_[j] & ~m & ~(1 << x)):
                if label[x] == label[y]:
                    st = (m | 1 << x | 1 << y, x, y)
                    if st not in seen: seen.add(st); stack.append(st)
    return best

def brute_directed(n, edges, label):
    adj = [[] for _ in range(n)]
    for u, v in edges: adj[u].append(v)
    best = 1
    def dfs(path, used):
        nonlocal best
        s = "".join(label[x] for x in path)
        if s == s[::-1]: best = max(best, len(path))
        for v in adj[path[-1]]:
            if not used >> v & 1: dfs(path + [v], used | 1 << v)
    for s in range(n): dfs([s], 1 << s)
    return best

rng = random.Random(5)
for _ in range(200):
    n = rng.randint(1, 7); edges = [[a, b] for a in range(n) for b in range(n) if a != b and rng.random() < 0.3]
    lab = "".join(rng.choice("ab") for _ in range(n))
    assert max_len_directed_sol(n, edges, lab) == brute_directed(n, edges, lab)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
adj[i] = bitmask láng giềng; lab[c] = bitmask nhãn c
khởi tạo: ({v}, v, v) mọi v; ({u,v}, u, v) mọi cạnh cùng nhãn
từ (m, i, j): với mỗi nhãn c:
    A = adj[i] & ~m & lab[c]; B = adj[j] & ~m & lab[c]
    mọi x ∈ A, y ∈ B, x != y -> (m | x | y, min, max)
best = max popcount(m); dừng khi best == n
```

**Checklist:**

- [ ] Có cả tâm lẻ (một đỉnh) và tâm chẵn (cạnh cùng nhãn)?
- [ ] `x ≠ y` khi mở rộng?
- [ ] Đánh dấu đỉnh đã dùng bằng `mask`?
- [ ] Chuẩn hoá `(i, j)` để giảm trạng thái?
- [ ] Đã so với liệt kê đường đơn?